# Mini-GPT exploration

Walks through:
1. loading a trained checkpoint
2. plotting attention patterns from each layer/head
3. plotting the loss curve from the MLflow run
4. a few qualitative samples

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.model import GPT, GPTConfig, CausalSelfAttention
from src.data import CharTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
# load ckpt
ckpt_path = '../out/tiny-shakespeare/ckpt.pt'
ckpt = torch.load(ckpt_path, map_location=device)
cfg = GPTConfig(**ckpt['config'])
model = GPT(cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
tok = CharTokenizer.load('../data/tiny-shakespeare/tokenizer.pkl')
print('vocab', tok.vocab_size, 'params', model.num_params())

## Attention pattern visualization

Hook into each `CausalSelfAttention` and capture the softmaxed attention weights.

In [ ]:
import math
import torch.nn.functional as F

captured = {}

def hook_attn(layer_idx):
    def _hook(module, inp, out):
        x = inp[0]
        B, T, C = x.size()
        qkv = module.c_attn(x)
        q, k, _ = qkv.split(module.n_embd, dim=2)
        q = q.view(B, T, module.n_head, module.head_dim).transpose(1, 2)
        k = k.view(B, T, module.n_head, module.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(module.head_dim))
        att = att.masked_fill(module.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        captured[layer_idx] = att.detach().cpu()
    return _hook

for i, blk in enumerate(model.blocks):
    blk.attn.register_forward_hook(hook_attn(i))

In [ ]:
prompt = 'ROMEO:\n'
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
with torch.no_grad():
    _ = model(ids)

fig, axes = plt.subplots(cfg.n_layer, cfg.n_head, figsize=(2*cfg.n_head, 2*cfg.n_layer))
for L in range(cfg.n_layer):
    for H in range(cfg.n_head):
        ax = axes[L, H] if cfg.n_layer > 1 else axes[H]
        ax.imshow(captured[L][0, H].numpy(), cmap='viridis')
        ax.axis('off')
        if L == 0:
            ax.set_title(f'h{H}')
fig.tight_layout()
fig.savefig('attention_patterns.png', dpi=120)
plt.show()

## Quick sample

In [ ]:
torch.manual_seed(0)
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=40)
print(tok.decode(out[0].tolist()))